[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/tutoring/session01/solutions/00_setup_check.ipynb)

# 00 · Setup check

Run each cell below with **Shift+Enter**, from top to bottom.

✅ If the last cell prints `You're ready! 🎉`, your setup works.

❌ If you see an error instead, paste its **last line** in the chat. In the meantime, open this notebook in **Colab** with the badge above so you can keep following along.

In [1]:
# On Colab, install Lightning (Colab already has PyTorch and TensorBoard). On your laptop, this does nothing.
import sys
import subprocess

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning"], check=True)

In [2]:
import matplotlib
import numpy as np
import pytorch_lightning as pl
import tensorboard
import torch

print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("lightning   :", pl.__version__)
print("tensorboard :", tensorboard.__version__)
print("numpy       :", np.__version__)
print("Running from:", sys.executable)

Python      : 3.12.16
torch       : 2.14.1
lightning   : 2.6.6
tensorboard : 2.21.0
numpy       : 2.5.5
Running from: /Users/abuchnick/miniforge3/envs/replearn/bin/python


The `Running from:` line shows **which Python** runs this notebook. On your laptop it should contain `replearn`, the conda environment you created. If it doesn't, click the kernel name in the top-right corner of VS Code and choose **replearn**.

Next: load the small MNIST file that every notebook today uses.

In [3]:
import io
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/SivanSchwartz/representation-learning-course/master/tutoring/data/mnist_small.npz"


def load_mnist():
    """The local copy if there is one (your laptop), otherwise download it (Colab)."""
    for path in ["../data/mnist_small.npz", "../../data/mnist_small.npz"]:
        if os.path.exists(path):
            return dict(np.load(path))
    return dict(np.load(io.BytesIO(urllib.request.urlopen(DATA_URL).read())))


data = load_mnist()
for name, array in data.items():
    print(f"{name:8} shape={array.shape}  dtype={array.dtype}")

X_train  shape=(6000, 28, 28)  dtype=uint8
y_train  shape=(6000,)  dtype=uint8
X_test   shape=(1000, 28, 28)  dtype=uint8
y_test   shape=(1000,)  dtype=uint8


In [4]:
# A tiny Lightning run: one batch of training, one of validation. It checks that the whole stack works.
import logging
import warnings

import torch.nn.functional as F
from torch import nn

warnings.filterwarnings("ignore", ".*does not have many workers.*")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

X = torch.tensor(data["X_train"][:256], dtype=torch.float32).reshape(-1, 784) / 255
y = torch.tensor(data["y_train"][:256], dtype=torch.long)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=64)


class Tiny(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 10)

    def training_step(self, batch, batch_idx):
        x, y = batch
        return F.cross_entropy(self.fc(x), y)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


trainer = pl.Trainer(fast_dev_run=True, accelerator="cpu", logger=False, enable_progress_bar=False, enable_model_summary=False)
trainer.fit(Tiny(), loader)
print("\nYou're ready! 🎉")


You're ready! 🎉


/Users/abuchnick/miniforge3/envs/replearn/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/abuchnick/miniforge3/envs/replearn/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
